<a href="https://colab.research.google.com/github/gyeongee/TIL/blob/main/%EA%B0%90%EC%84%B1%EB%B6%84%EC%84%9D_%EB%84%A4%EC%9D%B4%EB%B2%84%EC%98%81%ED%99%94%ED%8F%89%EC%A0%90%EB%8D%B0%EC%9D%B4%ED%84%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


### KoNLPy 사용하기

KoNIPy 설치 후 런타임 재시작할 필요 없습니다.

In [1]:
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 30.4 MB/s eta 0:00:00


In [2]:
from konlpy.tag import Okt
okt=Okt()
text="나는 오늘 자연어 처리를 공부하고 있습니다."
# Twitter 객체의 morphs() 메서드 사용
print(okt.morphs(text))

['나', '는', '오늘', '자연어', '처리', '를', '공부', '하고', '있습니다', '.']


In [3]:
print(okt.pos(text))

[('나', 'Noun'), ('는', 'Josa'), ('오늘', 'Noun'), ('자연어', 'Noun'), ('처리', 'Noun'), ('를', 'Josa'), ('공부', 'Noun'), ('하고', 'Josa'), ('있습니다', 'Adjective'), ('.', 'Punctuation')]


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


label 컬럼으로 감성이 분류가 되어있는 데이터를 사용하여 감성분석을 진행하겠습니다.

In [6]:
train_df=pd.read_csv('/content/ratings_train.txt',sep='\t',encoding="utf-8")
train_df.head(3)

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0


In [8]:
# 1이 긍정, 0이 부정 감성
train_df['label'].value_counts(normalize=True)

,proportion
label,
0,0.501153
1,0.498847


In [9]:
import re

train_df=train_df.fillna(' ')
# 정규표현식을 이용해 숫자를 공백으로 변경(정규표현식으로 \d는 숫자를 의미)
train_df['document']=train_df['document'].apply(lambda x:re.sub(r"\d+","",x))

# 테스트 데이터 세트 로딩하고 동일하게 NUll 및 숫자를 공백으로 변환
test_df=pd.read_csv('/content/ratings_test.txt',sep='\t',encoding="utf-8")
test_df=test_df.fillna(' ')
test_df['document']=test_df['document'].apply(lambda x:re.sub(r"\d+","",x))

In [10]:
# id 칼럼 삭제
train_df=train_df.drop('id',axis=1)
test_df=test_df.drop('id',axis=1)

각 문장을 한글 형태소 분석을 통해 형태소 단어로 토큰화하겠습니다.

한글 형태소 엔진은 SNS 분석에 적합한 Twitter 클래스를 이용하겠습니다.

In [12]:
from konlpy.tag import Twitter

twitter=Twitter()

def tw_tokenizer(text):
  # 입력 인자로 들어온 텍스트를 형태소 단어로 토큰화해 리스트 형태로 반환
  tokens_ko=twitter.morphs(text)
  return tokens_ko

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Twitter 객체의 morphs() 객체를 이용한 tokenizer를 사용,ngram_range는 (1,2)
tfidf_vect=TfidfVectorizer(tokenizer=tw_tokenizer,ngram_range=(1,2),min_df=3,max_df=0.9)
tfidf_vect.fit(train_df['document'])
tfidf_matrix_train=tfidf_vect.transform(train_df['document'])

# 로지스틱 회귀를 이용해 감성분석 분류 수행
lg_clf=LogisticRegression(random_state=0,solver='liblinear')

# 파라미터 C 최적화를 위해 GridSearchCV를 이용
params={'C':[1,3.5,4.5,5.5,10]}
grid_cv=GridSearchCV(lg_clf,param_grid=params,cv=3,scoring='accuracy',verbose=1)
grid_cv.fit(tfidf_matrix_train,train_df['label'])

print(grid_cv.best_params_,round(grid_cv.best_score_,4))


Fitting 3 folds for each of 5 candidates, totalling 15 fits
{'C': 3.5} 0.8591


In [14]:
from sklearn.metrics import accuracy_score

# 학습 데이터를 적용한 TfidfVectorizer를 이용해 테스트 데이터를 TF-IDF 값으로 피처 변환함
tfidf_matrix_test=tfidf_vect.transform(test_df['document'])

#classifier는 GridSearchCV에서 최적 파라미터로 학습된 classifier를 그대로 이용
best_estimator=grid_cv.best_estimator_
preds=best_estimator.predict(tfidf_matrix_test)

print(f'Logistic Regression 정확도:{accuracy_score(test_df['label'],preds)}')

Logistic Regression 정확도:0.86196
